# Particle Path Notebook
This notebook is for simulating the trajectory of charged particles through an ion thruster. [Magpylib](https://magpylib.readthedocs.io) is used for defining the static magnetic field of the thruster geometry. [PlasmaPy](https://docs.plasmapy.org) is used for running the charged particle simulation.

### Import Libraries
Along with magpylib and PlasmaPy, [astropy.units](https://docs.astropy.org/en/stable/units/index.html) is used for all physical units and numpy is used for arrays. [PyVista](https://pyvista.org/) will be used for visualization.

In [15]:
import astropy.units as u
import numpy as np

import magpylib as magpy
from plasmapy.particles import Particle
from plasmapy.plasma.grids import CartesianGrid
from plasmapy.simulation.particle_tracker.particle_tracker import ParticleTracker
from plasmapy.simulation.particle_tracker.save_routines import IntervalSaveRoutine
from plasmapy.simulation.particle_tracker.termination_conditions import (
    TimeElapsedTerminationCondition,
)

import pyvista as pv

## Thruster Magnet Geometry

### Define The Grid™
This `CartesianGrid` object stores a discrete cache of the magnetic (B) field in a grid of vertices. Using this, the simulation does not need to recalculate the field before each step, and can instead interpolate from the grid of precalculated points. The parameters for `CartesianGrid` below define the physical dimensions of the grid along with the number of points. If `num` is 10, the grid will be a 10x10x10 cube of 1000 points.

In [16]:
grid = CartesianGrid(-0.1 * u.m, 0.1 * u.m, num=50)

### Create Some Magnets
This next section will define the positions and polarities of each magnet. We use a nested for-loop to iterate over a radial and linear sequence.

In [17]:
RADIUS = 0.02
LAYER_SEPARATION = 0.03
POLARIZATION = 0.1
MAGNET_SIZE = (0.01, 0.01, 0.01)

RADIAL_COUNT = 4
LAYER_COUNT = 3

template_magnet = magpy.magnet.Cuboid(dimension=(0.01, 0.01, 0.01))
magnet_collection = magpy.Collection()

for layer_index in range(LAYER_COUNT):
    for radial_index in range(RADIAL_COUNT):
        array_index = layer_index * RADIAL_COUNT + radial_index

        magnet = template_magnet.copy()

        # The position at array_index is (x, y, z):
        theta = radial_index * 2 * np.pi / RADIAL_COUNT
        magnet.position = (
            np.sin(theta) * RADIUS,
            np.cos(theta) * RADIUS,
            layer_index * LAYER_SEPARATION - (LAYER_SEPARATION * LAYER_COUNT / 2),
        )

        # The polarization is negative if layer_index is odd
        magnet.polarization = (
            0.0,
            0.0,
            POLARIZATION if layer_index % 2 == 0 else -POLARIZATION
        )

        magnet_collection.add(magnet)


### Put the B field in The Grid™
Update the grid with vectors calculated from the magnets.

In [18]:
B = magnet_collection.getB(grid.grid) * u.T

grid.add_quantities(B_x=B[:,:,:,0], B_y=B[:,:,:,1], B_z=B[:,:,:,2])

## Particle Simulation
### Simulation Setup
First, we set up some initial conditions. `x0` defines the initial position and `v0` defines the initial velocity.

In [29]:
x0 = np.array([[0.002, 0, -0.070]], dtype = np.float32) * u.m
v0 = np.array([[0, 100, 2000]], dtype = np.float32) * u.m / u.s
particle = Particle("p+")

termination_condition = TimeElapsedTerminationCondition(0.00006 * u.second)
save_routine = IntervalSaveRoutine(0.0000001 * u.second)

The termination condition and save routine define how long and how many frames we will get out of the simulation.

### Running the Simulation
At last, we create and run the simulation and obtain results.

In [30]:
simulation = ParticleTracker(
    grid,
    save_routine=save_routine,
    termination_condition=termination_condition,
    verbose=False,
)

simulation.load_particles(x0, v0, particle)
simulation.run()

particle_trajectory = save_routine.results["x"][:, 0]
particle_trajectory

d:\Programming\Python\MagnaPy\env\Lib\site-packages\plasmapy\simulation\particle_tracker\particle_tracker.py:584: RuntimeWarning: Quantities should go to zero at edges of grid to avoid non-physical effects, but a value of 9.77E-05 T was found on the edge of the B_x array of grid 0. Consider applying a envelope function to force the quantities at the edge to go to zero.
  warnings.warn(
d:\Programming\Python\MagnaPy\env\Lib\site-packages\plasmapy\simulation\particle_tracker\particle_tracker.py:584: RuntimeWarning: Quantities should go to zero at edges of grid to avoid non-physical effects, but a value of 9.77E-05 T was found on the edge of the B_y array of grid 0. Consider applying a envelope function to force the quantities at the edge to go to zero.
  warnings.warn(
d:\Programming\Python\MagnaPy\env\Lib\site-packages\plasmapy\simulation\particle_tracker\particle_tracker.py:584: RuntimeWarning: Quantities should go to zero at edges of grid to avoid non-physical effects, but a value of 

<Quantity [[ 2.00630748e-03,  1.02687474e-04, -6.79617822e-02],
           [ 2.01870315e-03,  2.08845362e-04, -6.59237653e-02],
           [ 2.03588163e-03,  3.23856162e-04, -6.38862625e-02],
           [ 2.05553090e-03,  4.54503286e-04, -6.18497282e-02],
           [ 2.07179226e-03,  6.11594995e-04, -5.98150305e-02],
           [ 2.07455270e-03,  8.07595265e-04, -5.77836484e-02],
           [ 2.04368494e-03,  1.05472479e-03, -5.57580851e-02],
           [ 1.94675825e-03,  1.35831931e-03, -5.37423044e-02],
           [ 1.74111477e-03,  1.69939909e-03, -5.17407283e-02],
           [ 1.38433615e-03,  2.02538050e-03, -4.97579575e-02],
           [ 8.74444435e-04,  2.24430906e-03, -4.77940291e-02],
           [ 2.69224634e-04,  2.25306465e-03, -4.58450392e-02],
           [-3.03184293e-04,  2.00714334e-03, -4.39016409e-02],
           [-7.09520595e-04,  1.55159668e-03, -4.19542566e-02],
           [-8.80060077e-04,  9.93093941e-04, -3.99987698e-02],
           [-8.30144330e-04,  4.39597497

## Viewing the Results
Lastly, we need to put the results in something easy to view. Using PyVista, we can plot the magnets and the particle trajectory.

In [32]:
plotter = pv.Plotter()

# Magnets :)
magpy.show(magnet_collection, canvas=plotter)

# Particle Trajectory
plotter.add_lines(particle_trajectory * 1000, color = 'blue', width=2, connected=True)

plotter.show(jupyter_backend='client')

trigger(trigger__71)
trigger(trigger__72)
js_key = class
js_key = style
js_key = fluid
js_key = class
before: class = { 'rounded-circle': !P_0x22395e05690_10_show_ui }
(prefix=None) token {
has({ => {) = False
(prefix=None) token  
has(  =>  ) = False
(prefix=None) token '
has(' => ') = False
(prefix=None) token rounded
has(rounded => rounded) = False
(prefix=None) token -
has(- => -) = False
(prefix=None) token circle
has(circle => circle) = False
(prefix=None) token '
has(' => ') = False
(prefix=None) token :
has(: => :) = False
(prefix=None) token  
has(  =>  ) = False
(prefix=None) token !
has(! => !) = False
(prefix=None) token P_0x22395e05690_10_show_ui
has(P_0x22395e05690_10_show_ui => P_0x22395e05690_10_show_ui) = True
(prefix=None) translated P_0x22395e05690_10_show_ui
(prefix=None) token  
has(  =>  ) = False
(prefix=None) token }
has(} => }) = False
 => { 'rounded-circle': !P_0x22395e05690_10_show_ui }
after: class = { 'rounded-circle': !P_0x22395e05690_10_show_ui }
js_key =

Widget(value='<iframe src="http://localhost:51970/index.html?ui=P_0x22395e05690_10&reconnect=auto" class="pyvi…